# Tester les modèles du TP avec vos phrases

Ce notebook utilise les **checkpoints déjà entraînés** et leurs seuils de validation. Il ne lance aucun entraînement, ne lit aucun label test et n’exécute pas le notebook principal. Les architectures A, B, C, D et E sont prises en charge quand leurs fichiers sont présents.

Dans VS Code, choisir le Conda local pour les poids locaux, ou le **même serveur Colab T4** que l’entraînement pour les poids distants. Un serveur Colab neuf ne contient pas vos checkpoints : récupérer `results/registry.json`, `results/results.csv`, les checkpoints `.pt` et `.json` ou `.joblib`, `models/vocab.json` pour C et les ressources `models/hf/` pour B/D/E. Garder leur organisation dans un seul dossier projet. `TP2_ROOT` permet de désigner ce dossier si nécessaire.

Modifier ensuite `MODELE` et les phrases, puis relancer uniquement la prédiction. Le modèle sélectionné reste en mémoire entre deux saisies ; un changement de modèle libère le précédent. Les scores ne sont pas des garanties de compréhension de l’ironie, qui peut nécessiter du contexte.

### 1 — Préparer les dépendances

Le bloc Colab est identique à celui du notebook principal, avec les versions compatibles Python 3.13. Si une installation est nécessaire, son arrêt « Installation terminée » demande de redémarrer le noyau puis de recommencer. En Conda local, ce bloc n’installe rien. Aucun poids n’est téléchargé par ce notebook.

In [ ]:
from pathlib import Path
import os, sys, hashlib, base64, io, zipfile
IN_COLAB=bool(os.environ.get('COLAB_RELEASE_TAG')) or Path('/content').is_dir()
if IN_COLAB:
    print('TP2_COLAB_BOOTSTRAP_V2 — préparation compatible Python 3.13',flush=True)
    from importlib.metadata import version, PackageNotFoundError
    import subprocess
    # Copie autonome des dépendances, synchronisée avec requirements-colab.txt.
    COLAB_REQUIREMENTS = ['transformers==4.48.0', 'huggingface-hub==0.29.1', 'safetensors==0.5.3', 'tokenizers==0.21.0', 'gensim==4.4.0', 'arabert==1.0.1', 'pyarabic==0.6.15', 'emoji==1.4.2', 'sentencepiece==0.2.1', 'nbformat==5.9.2', 'reportlab==5.0.1', 'pymupdf==1.28.2', 'requests==2.32.3', 'pyflakes==3.0.1']
    missing=[]
    for specification in COLAB_REQUIREMENTS:
        package,wanted=specification.split('==',1)
        try: installed=version(package)
        except PackageNotFoundError: installed=None
        if installed!=wanted: missing.append(specification)
    if missing:
        print('Préparation Colab — Python',sys.version.split()[0],':',', '.join(missing),flush=True)
        # Ces extensions doivent avoir une wheel : éviter une compilation C++/Rust.
        command=[sys.executable,'-m','pip','install','--disable-pip-version-check',
                 '--progress-bar','off',
                 '--only-binary=safetensors,tokenizers,gensim,sentencepiece,pymupdf',*missing]
        installation=subprocess.run(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                    text=True,errors='replace')
        print(installation.stdout,flush=True)
        if installation.returncode:
            raise RuntimeError(f"Échec de pip (code {installation.returncode}). "
                               "Copier les lignes ERROR affichées ci-dessus pour le diagnostic. "
                               "Ne pas poursuivre les cellules scientifiques.")
        raise RuntimeError('Installation terminée : redémarre le noyau Colab puis relance depuis la première cellule.')
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'


### 2 — Retrouver les checkpoints et préparer le lecteur

Le dossier est déterminé à partir de `TP2_ROOT`, de la racine courante ou du dossier Colab de l’entraînement. Le petit paquet intégré fournit uniquement le code manquant ; il ne remplace pas les checkpoints. Il est vérifié par SHA256, extrait sous le projet et n’écrase pas les sources présentes. `PhraseTester` lit le registre et charge les poids seulement à la première prédiction. Les modules sont lisibles dans `src/predict.py`, `src/models.py` et `src/preprocess.py`.

In [ ]:
ROOT=Path(os.environ.get('TP2_ROOT',Path.cwd())).resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if IN_COLAB and not (ROOT/'results/registry.json').is_file() and 'TP2_ROOT' not in os.environ:
    candidates=[p for p in ROOT.iterdir() if p.is_dir() and (p/'results/registry.json').is_file()]
    ROOT=candidates[0] if len(candidates)==1 else Path('/content/TP2_ArSarcasme')
if not (ROOT/'results/registry.json').is_file():
    raise FileNotFoundError(f"Aucun registre dans {ROOT}. Choisir le même dossier distant que l'entraînement "
                            "ou récupérer results/registry.json, results/results.csv et les poids dans models/.")
SOURCE_ARCHIVE = 'UEsDBBQAAAAIAPdKUl0AAAAAAgAAAAAAAAAPAAAAc3JjL19faW5pdF9fLnB5AwBQSwMEFAAAAAgA90pSXSsRn7MgAwAAvAYAABEAAABzcmMvcHJlcHJvY2Vzcy5weY1UzW7TQBC+R8o7TOlhbZG4pMegAKEUiKAtauGUBGuxJ8lSe73aXRNKQeKG+Dlw4sAFCYRQDyB4hr5EIl6GsZ3mr5VCJNuT3ZlvZr75WYc7pyfy9ESfnoDiGtSRHSQSqjEYHXjmSAYQokqFgQhBJhafJMkhXIU4CUVPoJ4/9solEatEW9BYgYGNowqkUgRJiCG3fHqLcfJUlEu3Ws0taJCuFySxEhE6mrVHH6ujz+P31fGX8e+/n6p/f3aZWy492r/v728vKw+sVeZ6fWOjc3D55XA47HgksEqm1CKjne3dh6293QsMb7Q7Q797OUNu7jdvtrb8neYDUjJWezE/RKu5NM4xG31jdTb6wSps9H0qfT2Txn+m0luSxu/YKwIMsQf4PMvTDyLk0rH43Lr1cgnol8nkZ44UTyY65pF4gQ7bvb3FCt68VKIJuEKHQioQXNcjWSjHLaDW4ZFEMJhSARQXGsmrpToKTEluHmy1WlQmmV0aMKlSWsR5uQwEwnIrEmlASIuaXFGNIWI8tWTsFfiiR7qT6K/VgMswj56C4NqaobADh11i7uwCZTg7nuQ7l3P2adfq1Vp3MQ+NNtUyq49Jn2Qd0Ek3r1zB/N3L3pu8mn/oqIe9XpcoYvTkgZVLBeFhqiIRcIv+IR4tML4M3zFZhzCgV9Z/+SnBna+Y62lUEQ+oLuPXVGPmenlbROTFmXXNfFmKWBTZcY1+hpJDVbJhwQYLiHHUz4j6Z0juQ50oPxLy0DRu88jgUo+cj2halwwO1hrAVBSz80zP55Ub/mcmC+6LiStgiC7yOosXkMKlQ9J5uHdvexcmfhYAZtO3AuRge/8ClLM0G5Qm7/c1GpPRdj7bfJl4fbR+Lvka+0Sd4555nQedM5s2xPjN8Wbl1cvRr+xz1horbBzP7dQm+tRStdUW7cedYcd0L8Bfh7vcDCzvG6gDtYhFmcKkVWgn0wgL2kuJGiY6NMUfjGMh++DQ2u7ng7wRCh5hYGnYaY8Xlsb1Vk1A0RxL7YuaG/SDFM1Fm2sl27N9tWD333QvR+u0166PvnijD12ifCHwcukfUEsDBBQAAAAIAPdKUl2pj/0zFAYAADYRAAANAAAAc3JjL21vZGVscy5web1XzW7cNhC+G/A7EMhhqZSWva7jBl7oEDROe3B+0PhQIDAErjS7Yq0lFZJy1j70ddrkNfxiHZL63V27DlBUJ0kzmhnOzPfN6Bn55f6rvP+q77+SimtS3dpCSXKwIkZnsbmVGcmhqoUhJRCpLMyVuiYzslK5WAjQw9fx/p5YVUpbsuK22N9baLVCo7YoxZw0kg9e0jxYpbNWz9+3WlK2bzWXZqH0CrRpha9qq96qHMr9vbevfk8vzt8lpyd4//71+UX62/v3l4lzQtN0IUpI0yjWYFR5AzSK8YQgrfk0vTqcrJwJc1gsJvt7+3tZyY0hr6xFuVDyg1IllTJGP3UJ0dn+HsErhwVJUyGFTVNqoFywXKxaobtMXYFGP51SNCNOL/6SoLELIYFrit+w05NWcjOQnJ6wKZsLbpI3vDQQ9V4xBV+4zoPTgq24uR76LdVSWJMEg9SnMrZcFjQ4p0UURbH5XAPcAT2YRrEzALnLUEn/dA/sYAonUW+Rl1XBk2A3NmphV3ztvpwRDbbWktDiudeJazkwjF7qFZ1GzMv6xH603IrsHdinJPVGZXyeGnEHjONJkglvyzJhsJpDngu5TN4pCaxSRjhBcqlrYAvt4miS95SyePPcN2Gn6973XjDc8/aBDgKbHh2xivvXqcjXydEgd2JBOgMEkYP4IC7asw3j8RcQy8LGObc8zlR1m7alA2mUpp1iFD0UX2tCw+daYJ+nS83zlDqPIRmbX+ZaVaq27lyvwy2Nj1+Mg/dZSSYD7E0GyexMVVr9MWhel5Hp8ctorFnyW9BO67K3di4zxJ6+cCKKn7ATdvzilLWhxcdszm1WYHdqY31lox3uIVjZbZt6v+yYgeRzZAEJxmK/h8SO0NVe2ElJyD5Ha0ugDbdEgwZ37Z+Lm0YP1hUdfeHPz46j5/TAEWCM6HFZOTqKDl1iNh1CY+cOtDKtO5/CGQo/nbGjs7Pjq0bJCEkxxOfovhVPB+IMLbTiHbnSsBSYAJ3O68UCszOpYMIqcMVuEUSQDIEMIkpLcQ20gmHcTmdXKwxqcfHx8q3vBSSzzTIit+XYpZnzx8sHK9vBPRnz8bi7vGoBPB/2oM//A7S5ZoG60t7+FlM4KkzWscRiYpqLZIgZOoYeXUeoEnjSoXsEoZ5dHsVR46BJH+1ghYT9Q3jAQq9jU3C8mV5dMRzK6TXcpi31+Hg9gW+1syohT5ClnXA3SR96kb9n1wAVDqZQkxhJe1XRlZDJ9PHqP8P5nl0j3VX337ICEO6kqzDyiBb3f2vAUrhBLrABSV6TJnbCb7i0bn+4QRmxsLYQbxwCbYOvb21FaWKNd+6dPz7C2SDrYe6AhonYHCvOqppG280HEouQQYoMgFywmwS8Q5aOyxLebqgWqLURWJ4GzT6uxt5WKFZZXqYlyKUtkr6+Gy42+qifgmdNecOUTcagaZKxYcvXru2KYnv+jzskbAOxm/mYzxte1mB2bxsOgHSEkuBjyBrtxhC+C0FH7nSbeAwcFNT63eGNVncgf3WOnraRJT+dvnzS+N+krbCZfSdt/StlzfrJuzl2Z9/PYqG6T6CyAsu8KBW37rieJ3yd5y6sQfj/K8YexdfsP8RUCHUbVhl7CC4z4naPHT1Ns129jMrssSZG+aCDsa6XtYQn7r+Sr4Ct6tIK68rmk8jaXnWj5qnNnXS/SrH7m0orjBW5VgIez2ra/zEdOpe4JVQlxwxPDidscnAwiSJW4sZb+h8pkypZ3obm9xtuiMftty6kcOrm5Www2HA1kQuxjAuB7ST98twH/4xccGLvv1kgrujCoR/IzxcfyVxjgrFf/H+npxRN3MEMPogcd2rAxN//pcQNZp3wOqslcbuvwMd4NIqXgP8ethmuGBCbBHMT5lO5a0HHqIPO5l692c+7gT1tS2CQuQe4LjpUd98aDHek8aNPblf8kNZ+vdiiAyGr2uIfCNJq24Dp1t9hwISQlo6VWjA7nh82eWc06e5w3zwLdq5w6RlZScaPA80B7krT71JdKegOR+xR41GMgLJp20xYWb8oDyE+nE6uABR9PzyMuip4taibue7VsDXGo+kfUEsDBBQAAAAIAPdKUl3TPPFs9Q4AAFEvAAAOAAAAc3JjL3ByZWRpY3QucHm9Wltv5LYVfvevULdAqcnKsr0NgmCaCeBubHSRvRi2tyg6mQociZphrKEUkvIlW/epQID+kcRBkTToW1/9J7z/pueQukszs5sAGSRriTo8JM/1O0d69OjRMxHf30kmQuZETDnhkoUXWcqFVk6UO+cnjsqleZItJVVMeY6iAumoXDDpJMwRqWbzNL3wHz16tBPLdOVkVC8TPnf4Kkuldk7gdqe4XoTl1ZcqFeX1qkGRqvJKsp2dVPlMXHKZCl8xHbGY5ol2yeuzo+D8mHgO2SejTUTHzw//soXs/NXnRy+f/fXo9Cw4OTw9fP786Pmzsxc4KaaJYjCx3HE6h1OVdyJfZTcOVY7IyqGMiggG4L8sKsd0KsOlFYuWILk4lSsmVSmbw1ynL9KIJV59eZzKM/ZVjjp5mlCleMxDqnkqLM15esEE/5pJy9Vf4ZyK4TEX7DwX7CXTnnMs06+Z+BOjkeecaeARwnAxLZMsk2nIVDWVXeOfIEwYhaXwOZUs0OwaOL0+fR6cHnnOi6OX589evYTrnR0QljNxCM3nKy6ihM735kzq3TmYyS6VFG8u95/s6iuuNZNk5+T01YuT8zOY84ZQScaOS95+8/Cft984D9++/efD9w//e/jp7b8efny4cx7++/Dtww8PPz3c+aiItWT/BrIfkWjk7Tj1j8SW/VPm6CvG4GhKg93KkCrNQbKGafshOAFn+XWPExOG0/mSq4IcLkpWoWHUecYkT3MFjG53dnZCVKBzYnznHBZicmzYg6+8FkCbJw7o7/478CMmnNX93Srlkjl/AMdCVjkHzYKaLjkTggmNPinZgist0V+dS5rwyNiG8T7DGgzbCQIuuA4CV7Ek9hyZpqDDiF3ykE1epoKNxtUhkcJHAtALeqqL1yNfMpUml8wdtQktDyA1dl3cusVoKkFSYR5R4vC4oMBbn6uAXlIOJpIARwfMlTkkzHIy6rA3tgzczQ04ANzK8laXdl8O1Bac4hieq80tSWnEosFHksVwwiUcr5JZkoY0CTB2lVJjCYj2siksfAr83FpoezXZgMxADBAezTQUQkka6LRm0WCPP0k5SOfPNMnZkZSpdMlzhmEZfAy1X1pLlHKMkGhRToQRGcZAGl8yDZZXMZRM51KY9euDlmfHHfTPVu1rjwAVREi1V1jcjY8xm6w7XMxRuUOHOYYnL1N9nOYismeKyWlpxXSu0LDHzhtkdOs7rzVPYJbJLVEK4Q8uw1RoJijQJZ0kBXMlvf9B3N+p5sELJYepjBQcCjduzEG5ZrsSQqKJbK4xMi4WE5LrePfjnkUWMgAeWWSnherSHRSS+evD4+4+2nZYGxwIILDRu6sLmV7hktNZNQJ5w7lgN2iVocNF64A+12ylurKHpBrWCi12B0Y+bQ2Y9X32lQu8pkTQFSOzkfO79izFWFTR4A3QzNqKhg37NMuYiNw3BDZKxma3xPBnZNxgbzLriifV8AWkDzO8AHsR5ahdpxWN2z8SHwR1ACTjGOSsXXNsn+NB92c+PA+ArBH7R2i3CROWsAhGIvPBujatFXGVpYLPcdOFUqtwYbaLVyAWn12DWkEZt7URhGmSr4SaTI1gvFImXiUGrzy51zmT11x31nPryP+ManosQawuasArVyr+jnyFGf0Sg4lyu7wBqISgLzT+Y0Q6noOKmBClMU4TE8+YhkwSsWs3kmk2OZcgv9qACwgVwKnePZhELOQKE1YvmGwKJLDEe7qxv2AA7sJlCuEFd9hwymI95NnxJAcQnNNVb5NgCpNmfWWPe5ZTqAjIq0dGrKVQms4/6pGYv8ZZ7VVtBeCa/7Bj1op8CKLwP5VaAc5auuQ4IOi/lqalcR+itaBuw3MxSxs6tsr0zbuFbnKYh7looJYy/jqNTZqMFFbxeyAl2XULN7WOMastCzGo7NrUr4YQFqEPDpSwULdTeQMB+fomA0UB/DV4py26BvIxgg1CCimriTWCZVyADAyJnhO2cL51x8bJ4zRpHLNhmjaUqL1lvEceIyuw0yyhAMnIHkLT3d1mQgsTTCrb6ww8a3tHNkpWM7vKBMY+VhUBCFybSBa5YJeu3ffIK7AVOrYKUpHc2EjilzDIyhTcCYy1JSbUTyEosJCGRAr3RfzRceEhK/6c3ZS440VhtxxMR4gccQcwAtjxNBUYoBB3NF2zbbphnU/rcDAY9Nakh0HwNBxF1mOnpxX+6aOn0/u7MM+wpJcGKwFZ1EJJINgshzrlaQpwvAWZNF1lCG5N4jYbgyFwAfgTKHCr3uBK8xULIM20nMSymTQdrHMwazQglrYzWqdvmGu8sCkJNmVkCBqL+QKRQgM2dMIDvQ4gucOMjz5s5F+BSQILz9v1CcfEfnvCPI75tfFuP9Md514xTSGe0oF0hAE4sJNdYvPb6N2gZnuX5RI2hclcGDkXz8Gt39y2Z3ZODnbhFsSWQ/EIZn70YRffWp2vj2xGKi1uWWL83GZxyDVIClwCA78OPmqldKM9cNAp0TGPYtNUMT2JIJFkNh44RRHeba/FCNfIdtTjissq09MgA3wuwf3mbRU1QXsRNg3VeyuqtdGqreIipqzXhqAH9jslADyXaLFZqriJ7TBayLEYAaGYaFgvwpKW6EDgbcHVN0vYM96KhOMfjAVgKkJ3ZYupATZbrW12Hxg47jmHp4cDB2wm01bbqRfpe9gHQ//PylRrM0UPulXC2XbynsLMDeZfm3n77Gvpo4mh8LdwbLTcCqZG8yvIJlxTdYHqL5DKpLX6qLuwYmuW6gAdZNDTWlM0uPOmifTZ9k9R9Qrd5pK+Dbr+kkcRcMM80NOGta4JGlJ/IdNtLNqBA03HQ/vkpIHUhkXQwXID89zaqiewFcwSq/QSElQqgogvuA6AmGnegFgDLvcbEJ0NVlv6MjE5tuUbJHKoxSQ+hvhrO+MGVwC7W9KPXMZui9DVd2U0uG5kBNq651bFRA9K/CxAhzEnMg01z7lifLHUa52n1rlhFBjeIB7AuuayTY2mYYbX8ViD4FqUFcg20KCLHREG1IDPWl0B+TAYK+NNDWj07qFsk2Ua1thhaTa9XTUyfRaFKjIUs66pV8tiPRwkXFwoMlD9VezrfjmgirlLHFL20ut7Ndq8bMWs5weNm0177weWimWzy++qwvy8wWN6jvWZDeuICh9X2cMtlKgBxhSGikaJtmtAyUIvJ038sqEHg7+MRjYz19OJV5gSHEKAUNSEAGjrFZxvLsawv+nFrGuztrVmvZKLLNcBIGb0RKo1ZhWIHisTxW9rOwWxWZcxdlq9GgPkbNrrnjPH5pKJlpOPG/ZRy91OmaFNAZAS4BpQhrn1SzYti9YUViPleK98sPxg/1TcuKYYqnmVXOApPukYOvYNeOZ2lbm1GX0GIxzWw040LOVA3ivOb8LgJY/w7QUEQSgsywemV33/vXlnkZt1mE96Z2lsvZaeh2jWnKEecz5xDrZts0HdWD0XWAlhQ9mCr3hNdWfrTiwwUTjWVupW1+hnlyh6iY13KIqB3HYrDVU1PFAgrmyNEuP7HOZWlJVa951PJg2+cLNVNmf4Wqn99gikbG5Y9S6hfMvUlJDtIwJGg9Sb2P601+xRY/1TJCkuYmbeKptCulvaGpvDnhXanaRiwdx9zzRlrRE2/Wcgus6LzrYhnhpGY/Pv43rarDdrXfQ2Fc6GxNwH3AMQs7mt9w/6ZmZ/x2WrtNufapyoQntDkKX8VRWezdlF8AIIn86p200BFrsCIRjxYBroZQG7+9F07B30z9BG04MFW/mDihsb8ZA0UqjW0BoMZ7CGZoKA2whL1AmQQmyAinaQGe6Oe3aDTOQrJgHFFAyHl8efSVvKOLMPLhfRJHEl+eLq8d+nf/vi6gs1I+0X4y7aKtC+fvns6avPjoa3YlQV1RncVp0oV+05B1aQxhPs6tNx87izGXr6dECyldimcM4xigsWGc2wARCpd7AD13qqTZrudfV+uJkbR74NU/giIdawJXf3wCraB7QJw+YrCLd/7vcsYdd4VANQlMjQaNBiwsEp22u38teIVhTqW3w/1MQFpg9S2polKzocWM1h9zqalB2UDRaFvzTXbckjEGlgDVsfTnuAY705JSmUM2hRwHq630URMAiAK88SVmAIGBiW72DFWf5+67y4/x7w81c5hV1VreBCCuajoPs7cGjlYK43L2axzw2YwLzBxV4j1EOf+WsXqJqlRa3pfvABXI38BFQRlAUnVh/+kiaxWxnjWoYoNuA5KMz1rtmvmcFAN2sUay9YaNn2As9sYItnNH9ZmiamMlqCT1k1mvCL44ByyWwCVV2iiNWi6y4/WE193/cwG8zAJ/OVezDaW9kL74KxLOIrZau+kQ9SBNt1IU4auf3+yZbNbMoSdqdrYnz522xO+KvstuEMSyu2ze78m3dUDh7CLvLLIxcaBLaXhleT9Gpj/bgmp5tDRapfJwG/96iSHKiDApUBQKBJYPNG0dJoBZbhNCDZCjBhfw++hU8AOzDSAgPInxc8667TbypUshsqAKeqFkq1NI4ENfp7Mtrwon+6rMlbDJY4UnxVNjWOkwC3nC4Au8w2MCyLSCvihswJNk2CmEuFbfY1sn/85MPhirNbWA6uP+QBZdgzj9YIVlOeeGhw2N3nowLfGCRevQGzjls07Hj0xDwtPw5BVyKWD8EZ9mGSXoGKRmsMpXImcBw7dYYfBSwphJXdA+9JHx4YH9vf7mMQKDDB/PEJvltQTF5CAqGOvP9uAbUHlQvg5Yj7uwXoBQrsCEoRLkJt6kcVplDK4SsJLnKHxpCYlvd3w1lm2CC3+GzfDTcknDXu2TaxX9laBFv8clMBJu9vJ2uCLvDaZhL4a9Q7bsHPmgKymUyAy2gLF1uh+thhEBFmPqqolPTGzVDeppUy6s+y9Ww5qz3t00ldXPO43iJXpiazmbkcHVpk6FudN8R2Rsi46I61P5MyH0PgSw20dDIuyu7NjTEiKKzDgmIOOXGrVyKVI41ImU8BV9RvTIrvMa1fRaz4uhX7qVuWNN+n4gcmZGw/oNrAvpLiNqaSLVAQpAgC9Skm1Tne4RifTpx6d9uWLKCO+YhsSs7qb8XMQvOCMai7+Q0xMX41r1xEzW5HO/8HUEsBAhQAFAAAAAgA90pSXQAAAAACAAAAAAAAAA8AAAAAAAAAAAAAAIABAAAAAHNyYy9fX2luaXRfXy5weVBLAQIUABQAAAAIAPdKUl0rEZ+zIAMAALwGAAARAAAAAAAAAAAAAACAAS8AAABzcmMvcHJlcHJvY2Vzcy5weVBLAQIUABQAAAAIAPdKUl2pj/0zFAYAADYRAAANAAAAAAAAAAAAAACAAX4DAABzcmMvbW9kZWxzLnB5UEsBAhQAFAAAAAgA90pSXdM88Wz1DgAAUS8AAA4AAAAAAAAAAAAAAIABvQkAAHNyYy9wcmVkaWN0LnB5UEsFBgAAAAAEAAQA8wAAAN4YAAAAAA=='
payload=base64.b64decode(SOURCE_ARCHIVE)
assert hashlib.sha256(payload).hexdigest()=='f8daa2833def488ab17bcd6ede3aa800c6d54016e7c3a49c87607702e0fee5d0'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for entry in archive.infolist():
        target=(ROOT/entry.filename).resolve()
        if not target.is_relative_to(ROOT): raise ValueError('Chemin intégré invalide.')
        if not target.exists():
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(archive.read(entry))
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.predict import PhraseTester
from IPython.display import display
tester=PhraseTester(ROOT)
print('Dossier :',ROOT,'— appareil :',tester.device)


### 3 — Choisir le modèle

Le tableau affiche la clé du modèle, sa famille, sa graine, son F1 de validation et la présence de son fichier. Le choix initial reprend `decision.json` s’il est disponible, sinon le meilleur score de validation disponible hors ablations F. Pour comparer une autre architecture, copier sa clé exacte dans `MODELE` entre guillemets. `tester.refresh()` relit les fichiers après de nouveaux entraînements.

In [ ]:
display(tester.list_models())
MODELE=tester.default_key()
# Exemple de choix manuel : MODELE = 'E1_AraBERT_mono_seed42'
print('Modèle choisi :',MODELE)

### 4 — Insérer vos phrases

Écrire une phrase arabe par ligne entre les triples guillemets. Emojis, négations, dialecte et répétitions peuvent être conservés : le lecteur applique le même prétraitement que la famille entraînée. Comme pendant l’entraînement, la longueur est limitée à 64 tokens ; les textes plus longs sont tronqués. Les exemples sont des entrées de démonstration, sans labels de référence ni mesure de qualité.

In [ ]:
PHRASES = """يا سلام، الإنترنت انقطع مرة أخرى، خدمة رائعة فعلاً 😂
الجو جميل اليوم وسأذهب للمشي.
شكراً على التأخير ساعتين، فعلاً تحترمون وقت الناس!
لم يعجبني هذا الفيلم."""
phrases=[s.strip() for s in PHRASES.splitlines() if s.strip()]
print(len(phrases),'phrases prêtes à tester.')

### 5 — Afficher les prédictions

Le lecteur recharge le checkpoint exact et son seuil enregistré, puis utilise `eval()` et `inference_mode()`. Il affiche le score, la règle de décision et la prédiction. **B2 est particulier** : son score est P(sentiment négatif), mais sa décision dépend de l’argmax parmi les trois sentiments, comme dans le TP ; elle n’est pas obtenue en comparant cette probabilité à 0,5. Les autres scores sont comparés au seuil de validation sauvegardé. Aucun seuil n’est réglé sur les phrases saisies.

In [ ]:
resultats=tester.predict(phrases,key=MODELE)
display(resultats)

### 6 — Saisie interactive facultative

Exécuter cette cellule pour taper une nouvelle phrase à la demande. La même instance du modèle est réutilisée. Une entrée vide est ignorée. Pour revenir à la liste, modifier la cellule 4 et relancer la cellule 5.

In [ ]:
phrase=input('Saisir une phrase arabe : ').strip()
if phrase:
    display(tester.predict(phrase,key=MODELE))
else:
    print('Aucune phrase saisie.')